<a href="https://colab.research.google.com/github/Abish-Devcraft/DSGP-GRP11-SPATIUM/blob/furniture-classifier/notebooks/07_backbone_comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# connect Drive, restore furniture_v2, keep train and val images only
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, torch
from PIL import Image
from torchvision import models, transforms

DRIVE = "/content/drive/MyDrive/DSGP"
!unzip -q -o {DRIVE}/furniture_v2.zip -d /content
df = pd.read_csv(f"{DRIVE}/furniture_v2_final_manifest.csv")
df = df[df["split"].isin(["train", "val"])].reset_index(drop=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(len(df), "images |", device)

Mounted at /content/drive
2700 images | cuda


In [ ]:
# turn every image into a feature vector with a given model
tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def extract(net, bs=64):
    net = net.eval().to(device); out = []
    with torch.no_grad():
        for i in range(0, len(df), bs):
            x = torch.stack([tf(Image.open(p).convert("RGB")) for p in df["path"][i:i + bs]]).to(device)
            out.append(net(x).float().cpu().numpy())
    return np.concatenate(out)

In [ ]:
# extract features with 4 pretrained models (last layer removed)
r50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); r50.fc = torch.nn.Identity()
cnx = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1); cnx.classifier[2] = torch.nn.Identity()
eff = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1); eff.classifier = torch.nn.Identity()
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitb14")

feats = {}
for name, net in [("ResNet50", r50), ("ConvNeXt-Base", cnx), ("EfficientNetV2-S", eff), ("DINOv2 ViT-B/14", dino)]:
    feats[name] = extract(net)
    print(f"{name:18} {feats[name].shape}")
    net.cpu(); torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 103MB/s]


Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:07<00:00, 47.4MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 171MB/s]


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 304MB/s]


ResNet50           (2700, 2048)
ConvNeXt-Base      (2700, 1024)
EfficientNetV2-S   (2700, 1280)
DINOv2 ViT-B/14    (2700, 768)


In [ ]:
# same simple classifier on each feature set: overall val macro F1 and room-crop accuracy
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
tr, va = (df["split"] == "train").values, (df["split"] == "val").values
y, room = df["cls"].values, df["room_crop"].values.astype(bool)

rows = []
for name, X in feats.items():
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced"))
    clf.fit(X[tr], y[tr])
    p = clf.predict(X[va])
    rows.append({"backbone": name, "val macro F1": f1_score(y[va], p, average="macro"),
                 "room-crop acc": np.mean(p[room[va]] == y[va][room[va]]),
                 "shop-photo acc": np.mean(p[~room[va]] == y[va][~room[va]])})
res = pd.DataFrame(rows).sort_values("val macro F1", ascending=False).round(3)
res

,backbone,val macro F1,room-crop acc,shop-photo acc
3,DINOv2 ViT-B/14,0.686,0.650,0.851
1,ConvNeXt-Base,0.630,0.556,0.870
0,ResNet50,0.608,0.535,0.877
2,EfficientNetV2-S,0.590,0.520,0.831


**Decision:** DINOv2 ViT-B/14 gives the best frozen features: val macro F1 0.686 vs 0.608 for ResNet50, and room-crop accuracy 0.650 vs 0.535. Shop-photo accuracy is similar for all backbones (0.83–0.88), so the gain comes from real room photos. DINOv2 is chosen as the backbone for final training.